# Arsenal (P)layer Analysis

This notebook is a continuation of the A(r)senal Analysis project. It focuses on analyzing player performance metrics for Arsenal Football Club for the 2024-2025 season of the English Premier League and will use Python for data analysis and visualization.

In [18]:
# Importing Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split as tts
from sklearn.preprocessing import StandardScaler as SS
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import confusion_matrix as cm

import os
from dotenv import load_dotenv
import mysql.connector


In [19]:
# Connecting to MySQL Database
## Loading .env file
load_dotenv()

# Connecting to database on LAN
db_conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),  
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    use_pure=True # Forces use of pure Python implementation
)

## Verifying connection by fetching and printing table names
cursor = db_conn.cursor()
cursor.execute("SHOW TABLES")
results = cursor.fetchall()
for table in results:
    print(table)

('afc_23_24',)
('pl_fixtures',)
('pl_matches',)
('pl_matches_backup',)
('pl_player_stats',)
('pl_players',)
('pl_referees',)
('pl_table_23_24',)
('pl_table_24',)
('pl_teams',)
('pl_venues',)


In [20]:
# Extracting Match Data from Database

sql_query = """
SELECT
    m.match_date,
    ht.team_name AS Home_Team,
    at.team_name AS Away_Team,
    m.home_score AS Home_GF,
    m.away_score AS Away_GF,
    m.home_xg AS Home_xG,
    m.away_xg AS Away_xG
FROM pl_matches m
JOIN pl_teams ht ON m.home_team_id = ht.team_id
JOIN pl_teams at ON m.away_team_id = at.team_id
ORDER BY m.match_date;
"""

# Extracting into Pandas Dataframe

cursor = db_conn.cursor(dictionary=True) # dictionary=True returns results as dictionaries, ideal for Pandas
cursor.execute(sql_query)
# The pd.DataFrame() function handles the transfer of SQL results to a Pandas DataFrame
match_data_raw = pd.DataFrame(cursor.fetchall())
cursor.close()

# Converting match_date to datetime format
match_data_raw['match_date'] = pd.to_datetime(match_data_raw['match_date'])


# Converting Home_xG and Away_xG to numeric, coercing errors to NaN
match_data_raw['Home_xG'] = pd.to_numeric(match_data_raw['Home_xG'], errors='coerce')
match_data_raw['Away_xG'] = pd.to_numeric(match_data_raw['Away_xG'], errors='coerce')


# Validating Data Extraction
print("--- Raw Match Data Structure ---")
print(match_data_raw.info())
print("\n--- Raw Match Data Head ---")
print(match_data_raw.head())

--- Raw Match Data Structure ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   match_date  380 non-null    datetime64[ns]
 1   Home_Team   380 non-null    object        
 2   Away_Team   380 non-null    object        
 3   Home_GF     380 non-null    int64         
 4   Away_GF     380 non-null    int64         
 5   Home_xG     380 non-null    float64       
 6   Away_xG     380 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(2)
memory usage: 20.9+ KB
None

--- Raw Match Data Head ---
  match_date          Home_Team                Away_Team  Home_GF  Away_GF  \
0 2024-08-16  Manchester United                   Fulham        1        0   
1 2024-08-17    West Ham United              Aston Villa        1        2   
2 2024-08-17            Arsenal  Wolverhampton Wanderers        2        0   
3 2

In [21]:
# Converting match_data_raw to long format for easier analysis

home_df = match_data_raw.rename(columns={
    'Home_Team': 'Team',      # The Home Team is now "Team"
    'Away_Team': 'Opponent',  # The Away Team is now "Opponent"
    'Home_GF': 'GF',          # Home Goals now "Goals For"
    'Away_GF': 'GA',          # Away Goals now "Goals Against"
    'Home_xG': 'xG',
    'Away_xG': 'xGA'
})

# Add a label 
home_df['Location'] = 'Home'

away_df = match_data_raw.rename(columns={
    'Away_Team': 'Team',  # The Away Team is now "Team"
    'Home_Team': 'Opponent',      # The Home Team is now "Opponent"
    'Home_GF': 'GF',          # Home Goals now "Goals For"
    'Away_GF': 'GA',          # Away Goals now "Goals Against"
    'Home_xG': 'xG',
    'Away_xG': 'xGA'
})

away_df['Location'] = 'Away'

# Concatenating home and away dataframes
team_match_data = pd.concat([home_df, away_df], ignore_index=True)

# Sorting by match_date for chronological order
team_match_data = team_match_data.sort_values(by=['Team', 'match_date']).reset_index(drop=True)



In [22]:
# Preparing data for modeling

## Defining variables
window_size = 5

## Rolling averages for performance metrics
team_match_data['Rolling_xG_For'] = team_match_data.groupby('Team')['xG'].rolling(window=window_size).mean().reset_index(0, drop=True).shift(1)

team_match_data['Rolling_xG_Against'] = team_match_data.groupby('Team')['xGA'].rolling(window=window_size).mean().reset_index(0, drop=True).shift(1)

## Defining variable for opponent rolling averages
opponent_features = team_match_data[['match_date', 'Team', 'Rolling_xG_For', 'Rolling_xG_Against']]

### Merging rolling averages back into main dataframe
team_match_data = team_match_data.merge(
    opponent_features,
    left_on=['match_date', 'Opponent'],
    right_on=['match_date', 'Team'],
    suffixes=('_Team', '_Opponent'),
    how='left'
)

## Renaming columns for clarity
team_match_data.rename(columns={
    'Rolling_xG_For_Opponent': 'Opponent_Rolling_xG_For',
    'Rolling_xG_Against_Opponent': 'Opponent_Rolling_xG_Against'
}, inplace=True)

## Dropping redundant 'Team' column from opponent merge
team_match_data.drop(columns=['Team_Opponent'], inplace=True)

## Fixing 'Team' column name
team_match_data.rename(columns={'Team_Team': 'Team'}, inplace=True)

## Establish Target Variable
team_match_data['Target'] = (team_match_data['GF'] > team_match_data['GA']).astype(int)

## Encoding Location as Binary Variable
team_match_data['At_Home']= team_match_data['Location'].map({'Home': 1, 'Away': 0})

## Drop rows where rolling features are NA (the first 4 games for each team)
team_match_data.dropna(inplace=True)

## Verify how many rows are left
print(f"\nTotal rows ready for modeling: {len(team_match_data)}")

## Verifying the prepared data
arsenal_data = team_match_data[team_match_data['Team'] == 'Arsenal'].head(10)




Total rows ready for modeling: 678


In [23]:
## Distribution Check
print("Target Distribution (Win vs. Not Win):")
print(team_match_data['Target'].value_counts(normalize=True))

## Arsenal Data Sample for verification
print("\n--- Arsenal Data Sample ---")
print(arsenal_data.head(10))

arsenal_data.describe()

Target Distribution (Win vs. Not Win):
Target
0    0.584071
1    0.415929
Name: proportion, dtype: float64

--- Arsenal Data Sample ---
   match_date     Team           Opponent  GF  GA   xG  xGA Location  \
5  2024-09-28  Arsenal     Leicester City   4   2  4.4  0.3     Home   
6  2024-10-05  Arsenal        Southampton   3   1  2.8  0.6     Home   
7  2024-10-19  Arsenal        Bournemouth   2   0  1.8  0.7     Away   
8  2024-10-27  Arsenal          Liverpool   2   2  0.9  0.8     Home   
9  2024-11-02  Arsenal   Newcastle United   1   0  0.5  1.1     Away   
10 2024-11-10  Arsenal            Chelsea   1   1  1.5  1.5     Away   
11 2024-11-23  Arsenal  Nottingham Forest   3   0  1.2  0.3     Home   
12 2024-11-30  Arsenal    West Ham United   2   5  1.5  3.5     Away   
13 2024-12-04  Arsenal  Manchester United   2   0  2.1  0.2     Home   
14 2024-12-08  Arsenal             Fulham   1   1  0.2  1.8     Away   

    Rolling_xG_For_Team  Rolling_xG_Against_Team  Opponent_Rolling_xG_F

,match_date,GF,GA,xG,xGA,Rolling_xG_For_Team,Rolling_xG_Against_Team,Opponent_Rolling_xG_For,Opponent_Rolling_xG_Against,Target,At_Home
count,10,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,2024-11-06 04:48:00,2.100000,1.200000,1.690000,1.080000,1.798000,0.906000,1.574000,1.264000,0.600000,0.500000
min,2024-09-28 00:00:00,1.000000,0.000000,0.200000,0.200000,1.120000,0.600000,1.140000,1.040000,0.000000,0.000000
25%,2024-10-21 00:00:00,1.250000,0.000000,0.975000,0.375000,1.385000,0.725000,1.340000,1.085000,0.000000,0.000000
50%,2024-11-06 00:00:00,2.000000,1.000000,1.500000,0.750000,1.790000,0.870000,1.530000,1.230000,1.000000,0.500000
75%,2024-11-28 06:00:00,2.750000,1.750000,2.025000,1.400000,2.295000,0.930000,1.650000,1.380000,1.000000,1.000000
max,2024-12-08 00:00:00,4.000000,5.000000,4.400000,3.500000,2.420000,1.440000,2.400000,1.700000,1.000000,1.000000
std,NaN,0.994429,1.549193,1.216963,0.999778,0.524125,0.276815,0.364423,0.215572,0.516398,0.527046


In [ ]:
# Model Training and Evaluation Chunks
## Selecting features and target variable
features = [
    'Rolling_xG_For_Team',
    'Rolling_xG_Against_Team',
    'Opponent_Rolling_xG_For',
    'Opponent_Rolling_xG_Against',
    'At_Home'
]

X = team_match_data[features]
y = team_match_data['Target']

## Splitting data into training and testing sets (80-20 split)


In [24]:
print(team_match_data.head())

  match_date     Team          Opponent  GF  GA   xG  xGA Location  \
5 2024-09-28  Arsenal    Leicester City   4   2  4.4  0.3     Home   
6 2024-10-05  Arsenal       Southampton   3   1  2.8  0.6     Home   
7 2024-10-19  Arsenal       Bournemouth   2   0  1.8  0.7     Away   
8 2024-10-27  Arsenal         Liverpool   2   2  0.9  0.8     Home   
9 2024-11-02  Arsenal  Newcastle United   1   0  0.5  1.1     Away   

   Rolling_xG_For_Team  Rolling_xG_Against_Team  Opponent_Rolling_xG_For  \
5                 1.46                     0.90                     1.28   
6                 2.10                     0.86                     1.54   
7                 2.42                     0.80                     1.52   
8                 2.36                     0.60                     1.20   
9                 2.40                     0.62                     1.62   

   Opponent_Rolling_xG_Against  Target  At_Home  
5                         1.10       1        1  
6                     

In [25]:
# Closing the database connection
cursor.close()
db_conn.close()